# Chapter 3: Coding Attention Mechanisms

In [ ]:
import torch
import torch.nn as nn

In [ ]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
        [0.77, 0.25, 0.10],
        [0.05, 0.80, 0.55],
    ]
)

query = inputs[1]
query

In [ ]:
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

attn_scores_2

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
attn_weights_2_tmp

In [ ]:
attn_weights_2_tmp.sum()

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

def softmax_stable(x):
    """For my own L&D because it is fun.
    
    The naive softmax version overflows. We can normalise all vectors to [0, 1]."""
    e = torch.exp(x - x.max())
    return e / e.sum()

In [ ]:
attn_weights_2_naive = softmax_naive(attn_scores_2)
attn_weights_2_naive

In [ ]:
attn_weights_2_naive.sum()

In [ ]:
if torch.allclose(softmax_stable(attn_scores_2), softmax_naive(attn_scores_2)):
    print("Great success!")
else:
    print(softmax_stable(attn_scores_2))
    print(softmax_naive(attn_scores_2))

# Default diff tolerance on torch.allclose is e-05

In [ ]:
diff = (softmax_stable(attn_scores_2) - softmax_naive(attn_scores_2))
diff

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values

        return context_vec

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

In [ ]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

In [ ]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=1)

attn_weights.sum(dim=1)

In [ ]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

## 3.4.1 Computing the attention weights step by step

In [ ]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)

W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

query_2

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value

keys.shape

In [ ]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
attn_score_22

In [ ]:
attn_scores_2 = query_2 @ keys.T

Scaled-dot product attention

In [ ]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

In [ ]:
d_k

### Implementing a compact self-attention Python class

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v1(inputs)

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(789)

sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2(inputs)

In [ ]:
sa_v1(inputs)

In [ ]:
sa_v2.W_query.weight

if torch.allclose(sa_v2.W_query.weight, sa_v1.W_query.T):
    print("Success!")

In [ ]:
nn.Parameter(sa_v1.W_query.T)

### 3.5 Hiding future words with causal attention

With causal attention, a model can only factor in tokens that occur at or before the current token in the sequence.

In [ ]:
queries = sa_v2.W_query(inputs)

keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

attn_weights

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

In [ ]:
masked_simple = attn_weights * mask_simple
masked_simple

In [ ]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
masked_simple_norm